# 02 — Reinforcement Learning Basics

**Objectives**
- Build intuition for RL using a tiny environment.
- Collect trajectories and run a simple value-learning update loop.
- Visualize a short training curve.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip -q install numpy matplotlib
else:
    print("Running locally. Install once with: pip install -r requirements.txt")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1) Define a simple RL environment (LineWorld)

State space: positions `0..4` (goal at `4`).
Actions: `0` left, `1` right.

In [ ]:
class LineWorld:
    def __init__(self, n_states=5, max_steps=12):
        self.n_states = n_states
        self.max_steps = max_steps
        self.reset()

    def reset(self):
        self.state = 0
        self.steps = 0
        return self.state

    def step(self, action):
        self.steps += 1
        if action == 1:
            self.state = min(self.state + 1, self.n_states - 1)
        else:
            self.state = max(self.state - 1, 0)

        done = self.state == self.n_states - 1 or self.steps >= self.max_steps
        reward = 1.0 if self.state == self.n_states - 1 else -0.02
        return self.state, reward, done, {}

## 2) Q-learning training loop

In [ ]:
env = LineWorld(n_states=5, max_steps=12)
Q = np.zeros((env.n_states, 2))

alpha = 0.2
gamma = 0.95
epsilon = 0.2
episodes = 120

returns = []
transitions = []

for ep in range(episodes):
    s = env.reset()
    done = False
    ep_return = 0.0

    while not done:
        if np.random.rand() < epsilon:
            a = np.random.randint(2)
        else:
            a = int(np.argmax(Q[s]))

        s_next, r, done, _ = env.step(a)
        best_next = np.max(Q[s_next])
        Q[s, a] = Q[s, a] + alpha * (r + gamma * best_next - Q[s, a])

        transitions.append((s, a, r, s_next, done))
        s = s_next
        ep_return += r

    returns.append(ep_return)
    epsilon = max(0.05, epsilon * 0.99)

### Sample of collected observations / transitions

In [ ]:
print("First 8 transitions:")
for t in transitions[:8]:
    print(t)

print("\nLearned Q-table:")
print(np.round(Q, 3))

## 3) Plot training curve

In [ ]:
window = 10
moving_avg = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 3.5))
plt.plot(returns, alpha=0.4, label="Episode return")
plt.plot(range(window - 1, len(returns)), moving_avg, label=f"{window}-ep moving avg")
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title("Q-learning on LineWorld")
plt.legend()
plt.tight_layout()
plt.show()

## Quick exercises

1. Increase `n_states` to 8 and retrain. How does learning speed change?
2. Try `epsilon=0.5` and compare the early training curve.
3. Change step penalty from `-0.02` to `-0.1`. What policy does the agent learn?